In [13]:
import squigglepy as sq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from squigglepy.numbers import K, M, B

# Upgrade numpy and scipy to resolve potential version incompatibility issues
!pip install --upgrade numpy scipy

# Restart the runtime to ensure new package versions are loaded
exit()

sq.set_seed(42)
np.random.seed(42)
np.seterr(invalid='raise')  # Warn on operations involving NaN
N_SAMPLES = 5000

from chip_estimates_utils import (
    estimate_chip_sales,
    estimate_cumulative_chip_sales,
    aggregate_by_chip_type,
    interpolate_samples_to_calendar_quarters,
    compute_running_totals,
)

ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

# Loading data inputs

In [ ]:
# NVIDIA chip types
CHIP_TYPES = ['A100', 'A800', 'H100/H200', 'H800', 'H20', 'B200', 'B300']

# Colors for visualization
CHIP_COLORS = {
    'A100': 'lightcoral',
    'A800': 'sienna',
    'H100/H200': 'steelblue',
    'H800': 'firebrick',
    'H20': 'orange',
    'B200': 'mediumseagreen',
    'B300': 'seagreen',
}

# Hardware share of compute revenue (vs cloud/software)
# This is our main source of revenue uncertainty for NVIDIA
HARDWARE_SHARE = sq.to(0.96, 0.99, credibility=80)

In [ ]:
# Load revenue and price data from Google Sheets
revenue_df = pd.read_csv(
    "https://docs.google.com/spreadsheets/d/1Yhu87Rw--9tviAuBwg_luL3OFAFkdHdVfli6tN215Xk/export?format=csv&gid=0"
).set_index('Quarter')

prices_df = pd.read_csv(
    "https://docs.google.com/spreadsheets/d/1Yhu87Rw--9tviAuBwg_luL3OFAFkdHdVfli6tN215Xk/export?format=csv&gid=1819303346"
).set_index('Year')

QUARTERS = revenue_df.index.tolist()

# print(f"Loaded {len(QUARTERS)} quarters of data")
# print(revenue_df[['Compute revenue']].head())
# print()
# print(prices_df.head())

In [ ]:
# ==============================================
# CHIP SPECS FROM EPOCH.AI
# ==============================================
# Download chip specs including TOPS and TDP

import requests
import zipfile
import io

# Map notebook chip names to CSV chip names
CHIP_NAME_MAP = {
    'A100': 'A100',
    'A800': 'A800',
    'H100/H200': 'H100',  # Use H100 specs (H200 has same compute)
    'H800': 'H800',
    'H20': 'H20',
    'B200': 'B200',
    'B300': 'B300',
}

# Fallback specs extracted from chip_types.csv
# TOPS: 8-bit OP/s from CSV / 1e12
# TDP: TDP (W) column from CSV
FALLBACK_SPECS = {
    'A100':      {'tops': 624,  'tdp': 400},   # A100 80GB SXM
    'A800':      {'tops': 312,  'tdp': 400},   # A800 (export restricted variant)
    'H100/H200': {'tops': 1979, 'tdp': 700},   # H100 SXM
    'H800':      {'tops': 1979, 'tdp': 700},   # H800 (same as H100, export restricted)
    'H20':       {'tops': 296,  'tdp': 400},   # H20 (China-specific)
    'B200':      {'tops': 5000, 'tdp': 1200},  # B200 (Blackwell)
    'B300':      {'tops': 5000, 'tdp': 1400},  # B300 (Blackwell)
}

H100_TOPS = 1979  # Reference for H100-equivalent calculation

def load_chip_specs(chip_types_df, chip_name_map, fallback_specs):
    """Extract TOPS and TDP from CSV, with fallbacks for missing values."""
    specs = {}
    for notebook_name, csv_name in chip_name_map.items():
        row = chip_types_df[chip_types_df['Name'] == csv_name]
        if len(row) == 1:
            # Parse 8-bit OP/s (stored as float in scientific notation)
            tops_raw = row['8-bit OP/s'].values[0]
            tops = float(tops_raw) / 1e12 if pd.notna(tops_raw) else fallback_specs[notebook_name]['tops']

            # Parse TDP - try the linked column first
            tdp_col = 'TDP (W) (from ML Hardware (linked))'
            tdp_raw = row[tdp_col].values[0] if tdp_col in row.columns else None
            tdp = float(tdp_raw) if pd.notna(tdp_raw) else fallback_specs[notebook_name]['tdp']

            specs[notebook_name] = {'tops': tops, 'tdp': tdp}
        else:
            specs[notebook_name] = fallback_specs[notebook_name].copy()
            print(f"Warning: Using fallback specs for {notebook_name} (not found in CSV)")
    return specs

# Try to download and parse chip specs from epoch.ai
try:
    url = "https://epoch.ai/data/ai_chip_sales.zip"
    response = requests.get(url, timeout=10)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        with z.open("chip_types.csv") as f:
            chip_types_df = pd.read_csv(f)
            nvidia_chips_df = chip_types_df[chip_types_df['Designer'] == 'Nvidia']

    print(nvidia_chips_df[['Name', 'TDP (W) (from ML Hardware (linked))', '8-bit OP/s']].head(10))
    CHIP_SPECS = load_chip_specs(nvidia_chips_df, CHIP_NAME_MAP, FALLBACK_SPECS)
    print("\nLoaded Nvidia chip specs from epoch.ai CSV:")
except Exception as e:
    print(f"Warning: Could not download chip specs from epoch.ai: {e}")
    print("Using fallback Nvidia specs:")
    CHIP_SPECS = FALLBACK_SPECS.copy()

for chip, spec in CHIP_SPECS.items():
    print(f"  {chip}: TOPS={spec['tops']:.0f}, TDP={spec['tdp']:.0f}W")

# Price modeling setup

Load estimates (with confidence intevals) of chip prices, and setup a model of chip deflation over time.

In [ ]:
# Map chip types to their column names in the prices CSV
PRICE_COLUMN_MAP = {'H100/H200': 'H100'}

# Fallback prices if not found in CSV
FALLBACK_PRICES = {
    'A100': (10*K, 15*K), 'A800': (10*K, 15*K), 'H100/H200': (20*K, 30*K),
    'H800': (20*K, 30*K), 'H20': (10*K, 15*K), 'B200': (33*K, 42*K), 'B300': (33*K, 42*K)
}

# Build base price distributions (from first available year for each chip)
def get_price_dist_for_year(chip, year):
    """Get price distribution for a chip in a given year."""
    csv_chip_name = PRICE_COLUMN_MAP.get(chip, chip)
    low_col, high_col = f'{csv_chip_name} low', f'{csv_chip_name} high'

    if low_col in prices_df.columns and high_col in prices_df.columns:
        if year in prices_df.index:
            low = prices_df.loc[year, low_col]
            high = prices_df.loc[year, high_col]
            if pd.notna(low) and pd.notna(high):
                return sq.to(low, high, credibility=80)

    return sq.to(*FALLBACK_PRICES.get(chip, (20*K, 30*K)), credibility=80)

# Find first year each chip has price data
def find_first_year_with_price(chip):
    """Find the first year with price data for a chip."""
    csv_chip_name = PRICE_COLUMN_MAP.get(chip, chip)
    low_col = f'{csv_chip_name} low'

    if low_col in prices_df.columns:
        for year in sorted(prices_df.index):
            if pd.notna(prices_df.loc[year, low_col]):
                return year
    return min(prices_df.index)  # fallback to first year

# Build base prices dict
BASE_YEAR = {chip: find_first_year_with_price(chip) for chip in CHIP_TYPES}
BASE_PRICES = {chip: get_price_dist_for_year(chip, BASE_YEAR[chip]) for chip in CHIP_TYPES}

print("Base prices (first year available for each chip):")
for chip in CHIP_TYPES:
    dist = BASE_PRICES[chip]
    print(f"  {chip} ({BASE_YEAR[chip]}): ${dist.x:,.0f} - ${dist.y:,.0f}")

In [ ]:
# ==============================================
# DEFLATION FACTORS
# ==============================================

def get_price_bounds(chip, year):
    """Get (low, high) price bounds for a chip in a given year, or None if unavailable."""
    csv_chip_name = PRICE_COLUMN_MAP.get(chip, chip)
    low_col, high_col = f'{csv_chip_name} low', f'{csv_chip_name} high'

    if low_col in prices_df.columns and high_col in prices_df.columns:
        if year in prices_df.index:
            low = prices_df.loc[year, low_col]
            high = prices_df.loc[year, high_col]
            if pd.notna(low) and pd.notna(high):
                return (low, high)
    return None

def get_price_year_for_quarter(quarter):
    """Get the calendar year to use for pricing a quarter."""
    start_date = revenue_df.loc[quarter, 'Start Date']
    return pd.to_datetime(start_date).year

def get_deflation_factor(quarter, chip):
    """Get deflation factor for a chip in a quarter (ratio of current price to base price)."""
    price_year = get_price_year_for_quarter(quarter)
    base_year = BASE_YEAR[chip]

    if price_year <= base_year:
        return 1.0

    base_bounds = get_price_bounds(chip, base_year)
    current_bounds = get_price_bounds(chip, price_year)

    if base_bounds and current_bounds:
        # For lognormal sq.to(low, high), geometric mean = sqrt(low * high)
        return np.sqrt((current_bounds[0] * current_bounds[1]) / (base_bounds[0] * base_bounds[1]))
    return 1.0

# Print deflation factors for reference
print("Deflation factors by year (ratio to base year):")
years = sorted(prices_df.index)
for chip in CHIP_TYPES:
    factors = {}
    for year in years:
        # Find a quarter in this year to test
        for q in QUARTERS:
            if get_price_year_for_quarter(q) == year:
                factors[year] = round(get_deflation_factor(q, chip), 3)
                break
    if factors:
        print(f"  {chip}: {factors}")

# Model sample setup

The chip estimate model requires sampling functions that return values for model parameters for each quarter: chipmaker revenue, distribution of revenue across chip types, and price of a chip type for a given quarter

In [ ]:
# ==============================================
# SAMPLING FUNCTIONS
# ==============================================

def sample_revenue(quarter):
    """Return base revenue for a quarter (no uncertainty applied here)."""
    return revenue_df.loc[quarter, 'Compute revenue'] * B

def sample_shares(quarter):
    """Sample chip share of revenue for a quarter."""
    return {chip: revenue_df.loc[quarter, f'{chip} share'] for chip in CHIP_TYPES}

def sample_base_price(chip):
    """Sample base price for a chip (from its first available year)."""
    return BASE_PRICES[chip] @ 1

def sample_revenue_uncertainty():
    """Sample hardware share (our main source of revenue uncertainty)."""
    return HARDWARE_SHARE @ 1

# Cache price distributions by (chip, year) to avoid recreating them on every sample
PRICE_DIST_CACHE = {}

def sample_price(quarter, chip):
    """Sample price for a chip in a quarter (for uncorrelated model)."""
    year = get_price_year_for_quarter(quarter)
    cache_key = (chip, year)
    if cache_key not in PRICE_DIST_CACHE:
        PRICE_DIST_CACHE[cache_key] = get_price_dist_for_year(chip, year)
    return PRICE_DIST_CACHE[cache_key] @ 1

### [PROPOSED] Generation-based price correlation

This is a proposed modeling approach that has not been implemented yet.

Chip prices are correlated using a generation-based matrix rather than a flat scalar. Same-generation chips (e.g. B200/B300) are highly correlated, and correlation decays exponentially across generations (`decay^k`), which guarantees the matrix is positive semi-definite.

- **Chip generations**: Ampere (A100, A800), Hopper (H100/H200, H800, H20), Blackwell (B200, B300)
- **Within same generation**: 0.95
- **Cross-generation**: `0.5^k` where k = number of generations apart

In [ ]:
CHIP_GENERATIONS = {
    'A100': 1, 'A800': 1,                  # Ampere
    'H100/H200': 2, 'H800': 2, 'H20': 2,  # Hopper
    'B200': 3, 'B300': 3,                  # Blackwell
}

SAME_GEN_CORR = 0.95
CROSS_GEN_DECAY = 0.5

# illustrative, not implemented yet
def build_price_correlation_matrix(chip_types, same_gen_corr=SAME_GEN_CORR, cross_gen_decay=CROSS_GEN_DECAY):
    """Build a correlation matrix where same-generation chips are
    highly correlated and correlation decays as decay^k across generations."""
    n = len(chip_types)
    matrix = np.eye(n)
    for i in range(n):
        for j in range(i + 1, n):
            gen_diff = abs(CHIP_GENERATIONS[chip_types[i]] - CHIP_GENERATIONS[chip_types[j]])
            if gen_diff == 0:
                corr = same_gen_corr
            else:
                corr = cross_gen_decay ** gen_diff
            matrix[i][j] = corr
            matrix[j][i] = corr
    return matrix.tolist()

PRICE_CORRELATION = build_price_correlation_matrix(CHIP_TYPES)

# Display the matrix
print("Price correlation matrix:")
print(f"{'':>12}", '  '.join(f'{c:>10}' for c in CHIP_TYPES))
for i, chip in enumerate(CHIP_TYPES):
    print(f'{chip:>12}', '  '.join(f'{PRICE_CORRELATION[i][j]:>10.2f}' for j in range(len(CHIP_TYPES))))

# Running the main model

The next cell runs a Monte Carlo simulation to estimate chip volumes over time by dividing revenue among chip types (based on production mix) and then dividing by chip price.

Chip prices for each chip type are presampled once and reused across every quarter. This ensures that uncertainty in chip prices is applied across quarters; otherwise, sampling these prices separately across every quarter means that much of the uncertainty over the cumulative total gets canceled out. Because the samples are sorted (e.g. the first chip count in each quarter's sample represents is based on the same price draw), the sum of samples across quarters models uncertainty in the total chips delivered across quarters.

Prices for a given chip designer (e.g. Nvidia) are also partially correlated across chip type; this similarly widens the confidence intervals on aggregate totals.

The results are stored in the form of arrays of samples of chip counts, keyed by chip type and quarter. We then use percentiles to find medians and confidence intervals.

In [ ]:
# ==============================================
# RUN CORRELATED SIMULATION
# ==============================================

PRICE_CORRELATION = 0.5  # Correlation between chip prices across types

quarterly_samples = estimate_cumulative_chip_sales(
    quarters=QUARTERS,
    chip_types=CHIP_TYPES,
    sample_revenue=sample_revenue,
    sample_shares=sample_shares,
    sample_base_price=sample_base_price,
    get_deflation_factor=get_deflation_factor,
    sample_revenue_uncertainty=sample_revenue_uncertainty,
    price_correlation=PRICE_CORRELATION,
    base_price_distributions=BASE_PRICES,
    n_samples=N_SAMPLES
)

# Aggregate to get cumulative totals by chip type
cumulative_samples = aggregate_by_chip_type(quarterly_samples)

In [ ]:
# ==============================================
# CUMULATIVE SUMMARY
# ==============================================

def print_cumulative_summary(cumulative_samples, chip_types, title="Cumulative Production"):
    """Print formatted summary of cumulative chip counts with percentiles."""
    print(f"\n{title}")
    print(f"{'Version':<12} {'p5':>12} {'p50':>12} {'p95':>12}")
    print("-" * 51)

    grand_total = None
    for chip in chip_types:
        arr = cumulative_samples[chip]
        if arr.sum() > 0:
            if grand_total is None:
                grand_total = np.zeros_like(arr)
            grand_total += arr
            print(f"{chip:<12} {int(np.percentile(arr, 5)):>12,} {int(np.percentile(arr, 50)):>12,} {int(np.percentile(arr, 95)):>12,}")

    if grand_total is not None:
        print("-" * 51)
        print(f"{'TOTAL':<12} {int(np.percentile(grand_total, 5)):>12,} {int(np.percentile(grand_total, 50)):>12,} {int(np.percentile(grand_total, 95)):>12,}")

print_cumulative_summary(cumulative_samples, CHIP_TYPES, "Cumulative Nvidia Chip Sales")

Currently, the samples are keyed by Nvidia fiscal quarter, which do not line up with calendar quarters (so any calendar quarter will be split across two fiscal quarters). To interpolate fiscal quarter data to calendar quarters, we perform a weighted draw from the two fiscal quarters it spans, weighted by the time overlap.

In [ ]:
# ==============================================
# CALENDAR QUARTER INTERPOLATION (SAMPLE-BASED)
# ==============================================
# Interpolate per-quarter fiscal results to calendar quarters,
# then compute running totals from those

# Only accumulate chips sold from this date onward
CUMULATIVE_START_DATE = '1/1/2022'

# Build quarter_dates from revenue_df
quarter_dates = {q: (revenue_df.loc[q, 'Start Date'], revenue_df.loc[q, 'End Date'])
                 for q in QUARTERS}

# Step 1: Interpolate per-quarter samples to calendar quarters
calendar_quarterly_samples = interpolate_samples_to_calendar_quarters(quarterly_samples, quarter_dates)

# Filter to only include calendar quarters starting on or after CUMULATIVE_START_DATE
cutoff = pd.to_datetime(CUMULATIVE_START_DATE)
def _cq_start_date(cq):
    """Parse 'Q1 2024' -> start date of that calendar quarter."""
    q_num, year = int(cq[1]), int(cq.split()[1])
    month = {1: 1, 2: 4, 3: 7, 4: 10}[q_num]
    return pd.Timestamp(year, month, 1)

calendar_quarterly_samples = {
    cq: samples for cq, samples in calendar_quarterly_samples.items()
    if _cq_start_date(cq) >= cutoff
}

# Step 2: Compute running totals from calendar quarters
calendar_running_totals_samples = compute_running_totals(calendar_quarterly_samples)

In [ ]:
# ==============================================
# MULTI-METRIC CUMULATIVE RUNNING TOTALS
# ==============================================
# H100e compute, unit count, and total chip power by calendar quarter

def compute_multi_metric_samples(calendar_running_totals_samples, chip_specs, h100_tops=1979):
    """
    Compute cumulative H100e, units, and power for each calendar quarter.
    Preserves correlations by operating on full sample arrays.
    """
    results = {}
    for cq, chip_samples in calendar_running_totals_samples.items():
        n_samples = len(next(iter(chip_samples.values())))

        h100e_total = np.zeros(n_samples)
        units_total = np.zeros(n_samples)
        power_total_w = np.zeros(n_samples)

        for chip, samples in chip_samples.items():
            if chip in chip_specs:
                tops = chip_specs[chip]['tops']
                tdp = chip_specs[chip]['tdp']

                h100e_total += samples * (tops / h100_tops)
                units_total += samples
                power_total_w += samples * tdp

        results[cq] = {
            'h100e': h100e_total,
            'units': units_total,
            'power_mw': power_total_w / 1e6,
        }
    return results

# Compute metrics
multi_metric_samples = compute_multi_metric_samples(calendar_running_totals_samples, CHIP_SPECS, H100_TOPS)

# Display results
print("Cumulative Running Totals by Calendar Quarter: All Metrics")
print("=" * 95)
print(f"{'Quarter':<10} | {'H100e (millions)':^28} | {'Units (millions)':^28} | {'Power (GW)':^21}")
print(f"{'':10} | {'p5':>8} {'p50':>9} {'p95':>9} | {'p5':>8} {'p50':>9} {'p95':>9} | {'p5':>5} {'p50':>7} {'p95':>7}")
print("-" * 95)

for cq in multi_metric_samples:
    m = multi_metric_samples[cq]
    h5, h50, h95 = [np.percentile(m['h100e'], p) / 1e6 for p in [5, 50, 95]]
    u5, u50, u95 = [np.percentile(m['units'], p) / 1e6 for p in [5, 50, 95]]
    # Convert MW to GW for display
    p5, p50, p95 = [np.percentile(m['power_mw'], p) / 1e3 for p in [5, 50, 95]]

    print(f"{cq:<10} | {h5:>8.2f} {h50:>9.2f} {h95:>9.2f} | "
          f"{u5:>8.2f} {u50:>9.2f} {u95:>9.2f} | "
          f"{p5:>5.2f} {p50:>7.2f} {p95:>7.2f}")

In [ ]:
# ==============================================
# CUMULATIVE STACKED BAR CHART (H100-EQUIVALENT)
# ==============================================

quarters = list(quarterly_samples.keys())

# Get median values for each chip type, converted to H100-equivalents
chip_medians = {}
for chip in CHIP_TYPES:
    quarterly_medians = [np.median(quarterly_samples[q][chip]) for q in quarters]
    h100e_ratio = CHIP_SPECS[chip]['tops'] / H100_TOPS
    chip_medians[chip] = np.array(quarterly_medians) * h100e_ratio

# Calculate cumulative sums
chip_cumulative = {chip: np.cumsum(chip_medians[chip]) for chip in CHIP_TYPES}

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
x = np.arange(len(quarters))
width = 0.6

bottom = np.zeros(len(quarters))
for chip in CHIP_TYPES:
    ax.bar(x, chip_cumulative[chip], width, label=chip, bottom=bottom, color=CHIP_COLORS[chip])
    bottom += chip_cumulative[chip]

ax.set_ylabel('Cumulative H100-Equivalent Units', fontsize=12)
ax.set_xlabel('Quarter', fontsize=12)
ax.set_title('Cumulative NVIDIA Sales by Chip Type\n(H100-Equivalent Units)', fontsize=14, pad=20)
ax.set_xticks(x)
ax.set_xticklabels(quarters, rotation=45, ha='right')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, p: f'{y/1e6:.1f}M'))
ax.grid(True, alpha=0.3, axis='y')
ax.legend(fontsize=10, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================
# CORRELATION SENSITIVITY ANALYSIS
# ==============================================
# Run with several correlation values to see how price correlation affects CIs

CORRELATION_VALUES = [0.0, 0.3, 0.5, 0.6, 0.9, 0.99]
latest_cq = list(multi_metric_samples.keys())[-1]

# Collect H100e percentiles for each correlation level
results = {}
for rho in CORRELATION_VALUES:
    if rho == PRICE_CORRELATION:
        # Reuse existing correlated simulation
        m = multi_metric_samples[latest_cq]
    else:
        qs = estimate_cumulative_chip_sales(
            quarters=QUARTERS,
            chip_types=CHIP_TYPES,
            sample_revenue=sample_revenue,
            sample_shares=sample_shares,
            sample_base_price=sample_base_price,
            get_deflation_factor=get_deflation_factor,
            sample_revenue_uncertainty=sample_revenue_uncertainty,
            price_correlation=rho,
            base_price_distributions=BASE_PRICES,
            n_samples=N_SAMPLES,
        )
        cq_interp = interpolate_samples_to_calendar_quarters(qs, quarter_dates)
        cq_interp = {cq: s for cq, s in cq_interp.items() if _cq_start_date(cq) >= cutoff}
        cq_running = compute_running_totals(cq_interp)
        mm = compute_multi_metric_samples(cq_running, CHIP_SPECS, H100_TOPS)
        m = mm[latest_cq]

    p5, p50, p95 = [np.percentile(m['h100e'], p) / 1e6 for p in [5, 50, 95]]
    results[rho] = (p5, p50, p95)

# Print comparison table
print(f"Cumulative H100e (millions) — {latest_cq}")
print("=" * 95)
print(f"{'Model':<22} | {'p5':>7} {'p50':>8} {'p95':>8} | {'CI width':>9} | {'p5 vs p50':>10} {'p95 vs p50':>11}")
print("-" * 95)

for rho in CORRELATION_VALUES:
    p5, p50, p95 = results[rho]
    width = p95 - p5
    p5_pct = (p5 - p50) / p50 * 100
    p95_pct = (p95 - p50) / p50 * 100
    label = f"ρ = {rho}"
    if rho == PRICE_CORRELATION:
        label += " (default)"
    print(f"{label:<22} | {p5:>7.2f} {p50:>8.2f} {p95:>8.2f} | {width:>9.2f} | {p5_pct:>+9.1f}% {p95_pct:>+10.1f}%")

# Summary: how much wider is the default vs uncorrelated
w_default = results[PRICE_CORRELATION][2] - results[PRICE_CORRELATION][0]
w_zero = results[0.0][2] - results[0.0][0]
print(f"\nDefault correlation widens 90% CI by {w_default - w_zero:.2f}M H100e vs uncorrelated ({(w_default/w_zero - 1)*100:.0f}%)")

In [ ]:
# ==============================================
# QoQ GROWTH RATE OF H100e SHIPMENTS (FLOW)
# ==============================================
# Compute per-quarter H100e flow from fiscal quarter samples, then QoQ growth
# rates with 90% CIs. Each sample traces a consistent world (same prices/
# hardware share), so growth rates computed sample-by-sample preserve correlations.

# Compute per-fiscal-quarter H100e flow
h100e_flow = {}
for quarter in QUARTERS:
    flow = np.zeros(N_SAMPLES)
    for chip in CHIP_TYPES:
        if chip in CHIP_SPECS:
            flow += quarterly_samples[quarter][chip] * (CHIP_SPECS[chip]['tops'] / H100_TOPS)
    h100e_flow[quarter] = flow

# Print per-quarter flow levels for context
print("Per-Quarter H100e Shipments (flow, thousands) - Fiscal Quarters")
print(f"{'Quarter':<10} {'p5':>10} {'p50':>10} {'p95':>10}")
print("-" * 43)
for q in QUARTERS:
    p5, p50, p95 = [np.percentile(h100e_flow[q], p) / 1e3 for p in [5, 50, 95]]
    print(f"{q:<10} {p5:>10.1f} {p50:>10.1f} {p95:>10.1f}")

# Compute QoQ growth rates
print("\n\nQoQ Growth in H100e Shipments (per-quarter flow) - Fiscal Quarters")
print(f"{'Transition':<25} {'p5':>8} {'p50':>8} {'p95':>8} | {'p5':>8} {'p50':>8} {'p95':>8}  (annualized)")
print("=" * 90)

for i in range(1, len(QUARTERS)):
    prev_flow = h100e_flow[QUARTERS[i-1]]
    curr_flow = h100e_flow[QUARTERS[i]]

    valid = prev_flow > 0
    growth = np.full(N_SAMPLES, np.nan)
    growth[valid] = curr_flow[valid] / prev_flow[valid] - 1

    p5, p50, p95 = np.nanpercentile(growth, [5, 50, 95])
    # Annualized multiplier: (1+g)^4
    a5, a50, a95 = [(1 + g) ** 4 for g in [p5, p50, p95]]
    print(f"{QUARTERS[i-1]:>10} -> {QUARTERS[i]:<10} {p5:>7.1%} {p50:>8.1%} {p95:>8.1%} | {a5:>7.2f}x {a50:>7.2f}x {a95:>7.2f}x")

# Average QoQ growth (CAGR of flow, computed per-sample then percentiled)
n_transitions = len(QUARTERS) - 1
first_flow = h100e_flow[QUARTERS[0]]
last_flow = h100e_flow[QUARTERS[-1]]
valid = first_flow > 0
cagr_samples = np.full(N_SAMPLES, np.nan)
cagr_samples[valid] = (last_flow[valid] / first_flow[valid]) ** (1 / n_transitions) - 1

p5, p50, p95 = np.nanpercentile(cagr_samples, [5, 50, 95])
a5, a50, a95 = [(1 + g) ** 4 for g in [p5, p50, p95]]
print(f"\nAverage QoQ growth (CAGR over {n_transitions} quarters, {QUARTERS[0]} to {QUARTERS[-1]}):")
print(f"  Quarterly: {p50:.1%} ({p5:.1%} - {p95:.1%})")
print(f"  Annualized: {a50:.2f}x ({a5:.2f}x - {a95:.2f}x)")

In [ ]:
# ==============================================
# CSV EXPORTS: CUMULATIVE RUNNING TOTALS BY CHIP TYPE
# ==============================================
# Export calendar quarter running totals broken down by chip type

from datetime import datetime
from chip_estimates_utils import make_incomplete_note_fn

def get_calendar_quarter_dates(cal_q):
    """Return (start_date, end_date) strings for a calendar quarter like 'Q1 2024'."""
    parts = cal_q.split()
    q_num = int(parts[0][1])
    year = int(parts[1])
    if q_num == 1:
        return f"1/1/{year}", f"3/31/{year}"
    elif q_num == 2:
        return f"4/1/{year}", f"6/30/{year}"
    elif q_num == 3:
        return f"7/1/{year}", f"9/30/{year}"
    else:
        return f"10/1/{year}", f"12/31/{year}"

first_calendar_quarter = list(calendar_running_totals_samples.keys())[0]
first_start_date, _ = get_calendar_quarter_dates(first_calendar_quarter)

timestamp = datetime.now().strftime("%m-%d-%Y %H:%M")
generated_note = f"Estimates generated on: {timestamp}"

# Build incomplete note function from fiscal data coverage
nvidia_first_start = revenue_df['Start Date'].iloc[0]
nvidia_last_end = revenue_df['End Date'].iloc[-1]
get_incomplete_note = make_incomplete_note_fn(nvidia_first_start, nvidia_last_end, source_label='Nvidia')

rows = []
for cq in calendar_running_totals_samples:
    _, end_date = get_calendar_quarter_dates(cq)
    incomplete_note = get_incomplete_note(first_start_date, end_date)
    notes = generated_note
    if incomplete_note:
        notes = f"{incomplete_note}. {notes}"
    for chip in CHIP_TYPES:
        arr = calendar_running_totals_samples[cq][chip]
        if arr.sum() > 0:
            # Use display name (H100/H200 -> H100 for display)
            display_name = 'H100' if chip == 'H100/H200' else chip
            # Compute H100-equivalent
            h100e_factor = CHIP_SPECS[chip]['tops'] / H100_TOPS
            h100e_arr = arr * h100e_factor
            rows.append({
                'Name': f"{display_name} {first_calendar_quarter} to {cq}",
                'Chip manufacturer': 'Nvidia',
                'Start date': first_start_date,
                'End date': end_date,
                'Chip type': display_name,
                'Number of units (5th percentile)': int(np.percentile(arr, 5)),
                'Number of units (median)': int(np.percentile(arr, 50)),
                'Number of units (95th percentile)': int(np.percentile(arr, 95)),
                'Compute estimate in H100e (5th percentile)': int(np.percentile(h100e_arr, 5)),
                'Compute estimate in H100e (median)': int(np.percentile(h100e_arr, 50)),
                'Compute estimate in H100e (95th percentile)': int(np.percentile(h100e_arr, 95)),
                'Incomplete': 'checked' if incomplete_note else '',
                'Notes': notes,
            })

by_chip_df = pd.DataFrame(rows)
by_chip_df.to_csv('csv_export/nvidia_cumulative_by_chip.csv', index=False)
print(f"Exported {len(by_chip_df)} rows to csv_export/nvidia_cumulative_by_chip.csv")
# print(by_chip_df.head(10))

In [ ]:
# ==============================================
# CSV EXPORTS: FULL-NVIDIA AGGREGATE STATS
# ==============================================
# Export calendar quarter running totals with aggregate metrics across all chips
# Metrics: total units, H100e compute, power (MW)

# Get first calendar quarter dynamically
first_calendar_quarter = list(multi_metric_samples.keys())[0]
first_start_date, _ = get_calendar_quarter_dates(first_calendar_quarter)

rows = []
for cq in multi_metric_samples:
    _, end_date = get_calendar_quarter_dates(cq)
    m = multi_metric_samples[cq]

    incomplete_note = get_incomplete_note(first_start_date, end_date)
    notes = generated_note
    if incomplete_note:
        notes = f"{incomplete_note}. {notes}"

    rows.append({
        'Name': f"Nvidia total {first_calendar_quarter} to {cq}",
        'Chip manufacturer': 'Nvidia',
        'Start date': first_start_date,
        'End date': end_date,
        'Number of units (5th percentile)': int(np.percentile(m['units'], 5)),
        'Number of units (median)': int(np.percentile(m['units'], 50)),
        'Number of units (95th percentile)': int(np.percentile(m['units'], 95)),
        'Compute estimate in H100e (5th percentile)': int(np.percentile(m['h100e'], 5)),
        'Compute estimate in H100e (median)': int(np.percentile(m['h100e'], 50)),
        'Compute estimate in H100e (95th percentile)': int(np.percentile(m['h100e'], 95)),
        'Power in MW (5th percentile)': int(np.percentile(m['power_mw'], 5)),
        'Power in MW (median)': int(np.percentile(m['power_mw'], 50)),
        'Power in MW (95th percentile)': int(np.percentile(m['power_mw'], 95)),
        'Incomplete': 'checked' if incomplete_note else '',
        'Notes': notes,
    })

totals_df = pd.DataFrame(rows)
totals_df.to_csv('csv_export/nvidia_cumulative_totals.csv', index=False)
print(f"Exported {len(totals_df)} rows to csv_export/nvidia_cumulative_totals.csv")
# print(totals_df)

In [14]:
# ==============================================
# CSV EXPORT: FISCAL QUARTER CHIP TIMELINES
# ==============================================
# Export per-chip per-fiscal-quarter volumes (flow, not cumulative)

rows = []

for quarter in QUARTERS:
    start_date = revenue_df.loc[quarter, 'Start Date']
    end_date = revenue_df.loc[quarter, 'End Date']

    for chip in CHIP_TYPES:
        arr = quarterly_samples[quarter][chip]
        if arr.sum() > 0:
            u_p5, u_p50, u_p95 = [int(np.percentile(arr, p)) for p in [5, 50, 95]]

            h100e_factor = CHIP_SPECS[chip]['tops'] / H100_TOPS
            h_p5, h_p50, h_p95 = [int(p * h100e_factor) for p in [u_p5, u_p50, u_p95]]

            display_name = 'H100' if chip == 'H100/H200' else chip

            rows.append({
                'Name': f"{quarter} - {display_name}",
                'Chip manufacturer': 'Nvidia',
                'Start date': start_date,
                'End date': end_date,
                'Compute estimate in H100e (median)': h_p50,
                'H100e (5th percentile)': h_p5,
                'H100e (95th percentile)': h_p95,
                'Number of Units': u_p50,
                'Number of Units (5th percentile)': u_p5,
                'Number of Units (95th percentile)': u_p95,
                'Source / Link': '',
                'Notes': generated_note,
                'Chip type': display_name,
            })

fiscal_df = pd.DataFrame(rows)
fiscal_df.to_csv('csv_export/nvidia_fiscal_quarter_chip_timelines.csv', index=False)
print(f"Exported {len(fiscal_df)} rows to csv_export/nvidia_fiscal_quarter_chip_timelines.csv")
print(fiscal_df[['Name', 'Number of Units', 'Compute estimate in H100e (median)', 'Chip type']].to_string())

NameError: name 'QUARTERS' is not defined

In [ ]:
# ==============================================
# CSV EXPORT: CALENDAR QUARTER CHIP TIMELINES
# ==============================================
# Export per-chip per-calendar-quarter volumes (flow, not cumulative)
# Uses interpolated calendar quarter samples

rows = []

for cq in calendar_quarterly_samples:
    start_date, end_date = get_calendar_quarter_dates(cq)
    incomplete_note = get_incomplete_note(start_date, end_date)
    notes = generated_note
    if incomplete_note:
        notes = f"{incomplete_note}. {notes}"

    for chip in CHIP_TYPES:
        arr = calendar_quarterly_samples[cq][chip]
        if arr.sum() > 0:
            u_p5, u_p50, u_p95 = [int(np.percentile(arr, p)) for p in [5, 50, 95]]

            h100e_factor = CHIP_SPECS[chip]['tops'] / H100_TOPS
            h_p5, h_p50, h_p95 = [int(p * h100e_factor) for p in [u_p5, u_p50, u_p95]]

            display_name = 'H100' if chip == 'H100/H200' else chip

            rows.append({
                'Name': f"{cq} - {display_name}",
                'Chip manufacturer': 'Nvidia',
                'Start date': start_date,
                'End date': end_date,
                'Compute estimate in H100e (median)': h_p50,
                'H100e (5th percentile)': h_p5,
                'H100e (95th percentile)': h_p95,
                'Number of Units': u_p50,
                'Number of Units (5th percentile)': u_p5,
                'Number of Units (95th percentile)': u_p95,
                'Source / Link': '',
                'Notes': notes,
                'Chip type': display_name,
            })

calendar_df = pd.DataFrame(rows)
calendar_df.to_csv('csv_export/nvidia_calendar_quarter_chip_timelines.csv', index=False)
print(f"Exported {len(calendar_df)} rows to csv_export/nvidia_calendar_quarter_chip_timelines.csv")
print(calendar_df[['Name', 'Number of Units', 'Compute estimate in H100e (median)', 'Chip type']].to_string())

In [ ]:
# ==============================================
# CHRONOLOGICAL VIEW: FISCAL + CALENDAR INTERLEAVED
# ==============================================
# Pick a subset of quarters around a boundary to verify interpolation

from datetime import datetime

# Compute fiscal running totals for comparison
fiscal_running_totals_samples = compute_running_totals(quarterly_samples)

# Select a few interesting quarters to compare (around CY2024)
selected_fiscal = ['FY24Q4', 'FY25Q1', 'FY25Q2', 'FY25Q3', 'FY25Q4']
selected_calendar = ['Q1 2024', 'Q2 2024', 'Q3 2024', 'Q4 2024']

# Build timeline entries
timeline = []

for q in selected_fiscal:
    if q in QUARTERS:
        end_date = pd.to_datetime(revenue_df.loc[q, 'End Date'])
        timeline.append({
            'end_date': end_date,
            'label': q,
            'data': fiscal_running_totals_samples[q],
            'type': 'FISCAL'
        })

for cq in selected_calendar:
    if cq in calendar_running_totals_samples:
        parts = cq.split()
        q_num, year = int(parts[0][1]), int(parts[1])
        end_dates = {1: (3, 31), 2: (6, 30), 3: (9, 30), 4: (12, 31)}
        end_date = datetime(year, *end_dates[q_num])
        timeline.append({
            'end_date': end_date,
            'label': cq,
            'data': calendar_running_totals_samples[cq],
            'type': 'CALENDAR'
        })

# Sort chronologically
timeline.sort(key=lambda x: x['end_date'])

# Display with full chip breakdown
print("Chronological Comparison: Fiscal vs Calendar Quarter Running Totals")
print("=" * 80)

for entry in timeline:
    print(f"\n{entry['type']}: {entry['label']} (ends {entry['end_date'].strftime('%Y-%m-%d')})")
    print(f"  {'Chip':<12} {'p5':>12} {'p50':>12} {'p95':>12}")
    print(f"  {'-'*50}")

    total = np.zeros(N_SAMPLES)
    for chip in CHIP_TYPES:
        arr = entry['data'][chip]
        if arr.sum() > 0:
            total += arr
            print(f"  {chip:<12} {int(np.percentile(arr, 5)):>12,} {int(np.percentile(arr, 50)):>12,} {int(np.percentile(arr, 95)):>12,}")

    print(f"  {'-'*50}")
    print(f"  {'TOTAL':<12} {int(np.percentile(total, 5)):>12,} {int(np.percentile(total, 50)):>12,} {int(np.percentile(total, 95)):>12,}")

In [ ]:
# ==============================================
# UNCORRELATED MODEL COMPARISON
# ==============================================

# Run uncorrelated simulation (price sampled independently each quarter)
uncorrelated_samples = estimate_chip_sales(
    quarters=QUARTERS,
    versions=CHIP_TYPES,
    sample_revenue=sample_revenue,
    sample_shares=sample_shares,
    sample_price=sample_price,
    n_samples=1000 # not 10k because it was running slow
)

In [ ]:
cumulative_uncorrelated_samples = aggregate_by_chip_type(uncorrelated_samples)

print_cumulative_summary(cumulative_uncorrelated_samples, CHIP_TYPES, "Cumulative Nvidia Chip Sales (uncorrelated model)")

In [ ]:
# ==============================================
# SENSITIVITY: BUILT-IN PRICE CORRELATION vs POST-HOC SAMPLE CORRELATION
# ==============================================
# Compare two approaches to modeling correlation:
#   A) price_correlation=0.5 inside estimate_cumulative_chip_sales (correlates prices before computing units)
#   B) price_correlation=0 (independent prices), then correlate the resulting unit count samples post-hoc
#
# If they behave similarly, it means correlating outputs is a reasonable approximation
# to correlating inputs.

def correlate_running_totals_via_sq(running_totals, target_correlation, n_samples):
    """
    For each calendar quarter, fit lognormal distributions to each chip's samples,
    correlate them with sq.correlate, and re-sample.
    """
    result = {}
    for cq, chip_samples in running_totals.items():
        chips_with_data = [c for c in chip_samples if np.any(chip_samples[c] > 0)]

        if len(chips_with_data) < 2:
            result[cq] = chip_samples
            continue

        # Fit sq.to(p5, p95) lognormal to each chip's marginal samples
        dists = []
        for chip in chips_with_data:
            p5, p95 = np.percentile(chip_samples[chip], [5, 95])
            dists.append(sq.to(max(p5, 1), max(p95, 2)))  # clamp to avoid zero/negative

        correlated_dists = sq.correlate(tuple(dists), target_correlation)

        new_samples = {}
        for i, chip in enumerate(chips_with_data):
            new_samples[chip] = np.array(correlated_dists[i] @ n_samples)

        # Keep zero-chips as-is
        for chip in chip_samples:
            if chip not in new_samples:
                new_samples[chip] = chip_samples[chip]

        result[cq] = new_samples
    return result


# --- Approach A: Built-in price correlation = 0.5 ---
approach_a_samples = estimate_cumulative_chip_sales(
    quarters=QUARTERS,
    chip_types=CHIP_TYPES,
    sample_revenue=sample_revenue,
    sample_shares=sample_shares,
    sample_base_price=sample_base_price,
    get_deflation_factor=get_deflation_factor,
    sample_revenue_uncertainty=sample_revenue_uncertainty,
    price_correlation=0.5,
    base_price_distributions=BASE_PRICES,
    n_samples=N_SAMPLES,
)
a_interp = interpolate_samples_to_calendar_quarters(approach_a_samples, quarter_dates)
a_interp = {cq: s for cq, s in a_interp.items() if _cq_start_date(cq) >= cutoff}
a_running = compute_running_totals(a_interp)
a_metrics = compute_multi_metric_samples(a_running, CHIP_SPECS, H100_TOPS)

# --- Approach B: No price correlation, then correlate samples post-hoc via sq.correlate ---
approach_b_raw = estimate_cumulative_chip_sales(
    quarters=QUARTERS,
    chip_types=CHIP_TYPES,
    sample_revenue=sample_revenue,
    sample_shares=sample_shares,
    sample_base_price=sample_base_price,
    get_deflation_factor=get_deflation_factor,
    sample_revenue_uncertainty=sample_revenue_uncertainty,
    price_correlation=0,
    base_price_distributions=BASE_PRICES,
    n_samples=N_SAMPLES,
)
b_interp = interpolate_samples_to_calendar_quarters(approach_b_raw, quarter_dates)
b_interp = {cq: s for cq, s in b_interp.items() if _cq_start_date(cq) >= cutoff}
b_running = compute_running_totals(b_interp)

# Fit lognormals to each chip's marginal, correlate with sq.correlate, re-sample
b_running_correlated = correlate_running_totals_via_sq(b_running, 0.5, N_SAMPLES)
b_metrics = compute_multi_metric_samples(b_running_correlated, CHIP_SPECS, H100_TOPS)

# --- Compare ---
print("Built-in price correlation vs post-hoc sample correlation (ρ = 0.5)")
print("=" * 100)
print(f"{'Quarter':<10} | {'Approach A (price corr)':^30} | {'Approach B (post-hoc corr)':^30} | {'Diff':^20}")
print(f"{'':10} | {'p5':>8} {'p50':>9} {'p95':>9} | {'p5':>8} {'p50':>9} {'p95':>9} | {'CI width':^20}")
print("-" * 100)

for cq in list(a_metrics.keys()):
    ma = a_metrics[cq]
    mb = b_metrics[cq]
    a5, a50, a95 = [np.percentile(ma['h100e'], p) / 1e6 for p in [5, 50, 95]]
    b5, b50, b95 = [np.percentile(mb['h100e'], p) / 1e6 for p in [5, 50, 95]]
    a_width = a95 - a5
    b_width = b95 - b5
    width_diff_pct = (b_width / a_width - 1) * 100 if a_width > 0 else 0
    print(f"{cq:<10} | {a5:>8.2f} {a50:>9.2f} {a95:>9.2f} | {b5:>8.2f} {b50:>9.2f} {b95:>9.2f} | {width_diff_pct:>+8.1f}% width")

# Summary for latest quarter
latest = list(a_metrics.keys())[-1]
ma = a_metrics[latest]
mb = b_metrics[latest]
a5, a50, a95 = [np.percentile(ma['h100e'], p) / 1e6 for p in [5, 50, 95]]
b5, b50, b95 = [np.percentile(mb['h100e'], p) / 1e6 for p in [5, 50, 95]]
print(f"\nSummary for {latest}:")
print(f"  A (price corr):    p5={a5:.2f}M  p50={a50:.2f}M  p95={a95:.2f}M  CI width={a95-a5:.2f}M")
print(f"  B (post-hoc corr): p5={b5:.2f}M  p50={b50:.2f}M  p95={b95:.2f}M  CI width={b95-b5:.2f}M")
print(f"  CI width difference: {(b95-b5)/(a95-a5)*100 - 100:+.1f}%")